# Analyse des Taux de Consultations par Professionnels

Ce notebook analyse les consultations médicales depuis le schéma Gold.

**Analyses effectuées:**
1. Taux de consultations par spécialité médicale
2. Top 20 professionnels par nombre de consultations
3. Évolution temporelle des consultations par spécialité
4. Consultations moyennes par professionnel et spécialité
5. Statistiques globales
6. Répartition annuelle des consultations

In [ ]:
from pyspark.sql import SparkSession
import os

# Create Spark session
spark = SparkSession.builder \
    .appName("CHU - Analyse Consultations Professionnels") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark {spark.version} initialized")

In [ ]:
# Load Gold tables
data_base = os.getenv("DATA_BASE", "/home/jovyan/data")
gold_base = f"{data_base}/gold"

print("Loading Gold tables...")
fait_consultation = spark.read.parquet(f"{gold_base}/fait_consultation")
dim_professionnel = spark.read.parquet(f"{gold_base}/dim_professionnel")
dim_temps = spark.read.parquet(f"{gold_base}/dim_temps")

fait_consultation.createOrReplaceTempView("fait_consultation")
dim_professionnel.createOrReplaceTempView("dim_professionnel")
dim_temps.createOrReplaceTempView("dim_temps")

print(f"Tables loaded successfully")
print(f"- fait_consultation: {fait_consultation.count():,} rows")
print(f"- dim_professionnel: {dim_professionnel.count():,} rows")
print(f"- dim_temps: {dim_temps.count():,} rows")

## Analyse 1: Taux de Consultations par Spécialité Médicale

In [ ]:
query1 = """
SELECT
    p.nom_specialite,
    COUNT(*) as nb_consultations,
    COUNT(DISTINCT c.id_patient) as patients_uniques,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as pourcentage_consultations
FROM fait_consultation c
JOIN dim_professionnel p ON c.id_prof = p.id_prof
WHERE p.nom_specialite IS NOT NULL
GROUP BY p.nom_specialite
ORDER BY nb_consultations DESC
LIMIT 20
"""

result1 = spark.sql(query1)
print("\n" + "="*80)
print("Top 20 Spécialités Médicales par Nombre de Consultations")
print("="*80)
result1.show(20, truncate=False)

## Analyse 2: Top 20 Professionnels par Nombre de Consultations

In [ ]:
query2 = """
SELECT
    p.id_prof,
    p.nom,
    p.prenom,
    p.nom_specialite,
    COUNT(*) as nb_consultations,
    COUNT(DISTINCT c.id_patient) as patients_uniques,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) as taux_pourcentage
FROM fait_consultation c
JOIN dim_professionnel p ON c.id_prof = p.id_prof
GROUP BY p.id_prof, p.nom, p.prenom, p.nom_specialite
ORDER BY nb_consultations DESC
LIMIT 20
"""

result2 = spark.sql(query2)
print("\n" + "="*80)
print("Top 20 Professionnels Individuels")
print("="*80)
result2.show(20, truncate=False)

## Analyse 3: Évolution Temporelle par Spécialité (Top 10 par année)

In [ ]:
query3 = """
SELECT
    t.annee,
    p.nom_specialite,
    COUNT(*) as nb_consultations,
    COUNT(DISTINCT c.id_prof) as professionnels_actifs,
    COUNT(DISTINCT c.id_patient) as patients_uniques
FROM fait_consultation c
JOIN dim_professionnel p ON c.id_prof = p.id_prof
JOIN dim_temps t ON c.id_temps = t.id_temps
WHERE p.nom_specialite IS NOT NULL
GROUP BY t.annee, p.nom_specialite
ORDER BY t.annee, nb_consultations DESC
"""

result3 = spark.sql(query3)
print("\n" + "="*80)
print("Évolution Temporelle des Consultations par Spécialité")
print("="*80)
result3.show(50, truncate=False)

## Analyse 4: Consultations Moyennes par Professionnel et Spécialité

In [ ]:
query4 = """
SELECT
    p.nom_specialite,
    COUNT(DISTINCT p.id_prof) as nb_professionnels,
    COUNT(*) as nb_consultations_total,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT p.id_prof), 2) as consultations_moy_par_prof,
    COUNT(DISTINCT c.id_patient) as patients_uniques,
    ROUND(COUNT(DISTINCT c.id_patient) * 1.0 / COUNT(DISTINCT p.id_prof), 2) as patients_moy_par_prof
FROM fait_consultation c
JOIN dim_professionnel p ON c.id_prof = p.id_prof
WHERE p.nom_specialite IS NOT NULL
GROUP BY p.nom_specialite
ORDER BY consultations_moy_par_prof DESC
LIMIT 20
"""

result4 = spark.sql(query4)
print("\n" + "="*80)
print("Top 20 Spécialités par Consultation Moyenne par Professionnel")
print("="*80)
result4.show(20, truncate=False)

## Analyse 5: Statistiques Globales

In [ ]:
query5 = """
SELECT
    COUNT(*) as total_consultations,
    COUNT(DISTINCT id_prof) as total_professionnels,
    COUNT(DISTINCT id_patient) as total_patients,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT id_prof), 2) as consultations_moy_par_prof,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT id_patient), 2) as consultations_moy_par_patient
FROM fait_consultation
"""

result5 = spark.sql(query5)
print("\n" + "="*80)
print("Statistiques Globales")
print("="*80)
result5.show(truncate=False)

## Analyse 6: Répartition Annuelle des Consultations

In [ ]:
query6 = """
SELECT
    annee,
    COUNT(*) as nb_consultations,
    COUNT(DISTINCT id_patient) as patients_uniques,
    COUNT(DISTINCT id_prof) as professionnels_actifs,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as pourcentage_annuel
FROM fait_consultation
GROUP BY annee
ORDER BY annee
"""

result6 = spark.sql(query6)
print("\n" + "="*80)
print("Répartition Annuelle des Consultations")
print("="*80)
result6.show(20, truncate=False)

## Export des résultats en CSV (optionnel)

In [ ]:
# Optionnel: Exporter les résultats
output_base = f"{data_base}/exports/consultations_professionnels"

# result1.coalesce(1).write.mode("overwrite").csv(f"{output_base}/par_specialite", header=True)
# result2.coalesce(1).write.mode("overwrite").csv(f"{output_base}/top_professionnels", header=True)
# result4.coalesce(1).write.mode("overwrite").csv(f"{output_base}/moyennes_par_specialite", header=True)

print("Analyse terminée avec succès !")